Please go through the "building_strategies" notebook first before you go through this notebook


## Some Prebuilt Reporting ##

Lets first build the strategy described in that notebook, add it to a portfolio and run the portfolio

In [ ]:
import polars as pl
import numpy as np
import gambit as pq
from build_example_strategy import build_example_strategy

strategy = build_example_strategy()
strategy.run()

Many objects have functions that return pandas dataframes for ease of use.  Any function that returns a dataframe starts with df_ so its easy to tell which dataframes an object returns.

df_data is a useful function. This returns the market data, indicators, signal values and P&L at each market data bar.  The last column, i is the integer index of that bar, and can be used to query data in other dataframes or objects for that strategy.

Since we have daily pnl but minute bars a lot of the rows will have nans. So lets get EOD data (equity is not NaN in those rows)

In [ ]:
df_data = strategy.df_data()
df_data.filter(pl.col('equity').is_finite()).head()

You can also look at just the PNL or just the marketdata by themselves.

In [ ]:
strategy.df_pnl().tail()

We can look at orders and trades that were created during this run

In [ ]:
strategy.df_orders().head()

In [ ]:
strategy.df_trades().head()

In [ ]:
strategy.df_roundtrip_trades()

You can also look at the returns at the portfolio level (i.e. summing up several strategies)

In [ ]:
strategy.df_returns().tail()

We can get data as native Python objects as opposed to pandas dataframes.

In [ ]:
strategy.trades(start_date = np.datetime64('2023-01-13'), end_date = np.datetime64('2023-01-15'))

## Adding your Own Metrics ##

Each strategy may have metrics that you want to measure that are specific to that strategy.  To add these, you can use the Evaluator object which can make things easier.

To evaluate a strategy we use the evaluate returns function.

In [ ]:
strategy.evaluate_returns(plot = False);

What if we want to add some more metrics to this.  For example, lets say we want to add a metric that looks at how many long trades we had versus short trades.  We can do this using an Evaluator object.

In [ ]:
def compute_num_stopped_trades(trades):
    return len([trade for trade in trades if trade.order.reason_code == 'STOPPED_OUT'])

evaluator = pq.Evaluator(initial_metrics = {'trades' : strategy.trades()})

evaluator.add_metric('num_stopped_trades', compute_num_stopped_trades, dependencies = ['trades'])

evaluator.compute()

print(f'Stopped Trades: {evaluator.metric("num_stopped_trades")}')

The Evaluator takes care of dependency management so that if you want to compute a metric that relies on other metrics, it will compute the metrics in the right order.

Lets compute Maximum Adverse Execution for each trade.  MAE tells you the maximum loss each trade had during its lifetime. It's useful for figuring out where to put trailing stops. For example, if most of your profitable trades had a maximum loss during their life up to 5% but many losing trades had losses of 50% and 60%, it might make sense to place a trailing stop around 6% or 7% so you don't get stopped out of your profitable trades but get out of the losing ones quickly. See Jaekle and Tomasini, page 66 for details

In [ ]:
def compute_mae(rt_trades, c, timestamps):
    mae = np.full(len(rt_trades), np.nan)
    round_trip_pnl = np.full(len(rt_trades), np.nan)

    for i, rt in enumerate(rt_trades):
        _c = c[(timestamps >= rt.entry_timestamp) & (timestamps <= rt.exit_timestamp)]
        _mae = np.min(_c) / rt.entry_price - 1
        _mae = min(0, _mae)   # if we did not get a drawdown for this trade
        mae[i] = -_mae
        round_trip_pnl[i] = rt.net_pnl / rt.qty # Also store round trip pnl for this trade for plotting
    return mae, round_trip_pnl
        
contract_group = strategy.contract_groups[0]
evaluator = pq.Evaluator(initial_metrics = {'rt_trades' : strategy.roundtrip_trades(),
                                            'c' : strategy.indicator_values[contract_group.name].c,
                                            'timestamps' : strategy.timestamps})
evaluator.add_metric('mae', compute_mae, dependencies=['rt_trades', 'c', 'timestamps'])
evaluator.compute()

We could have easily run the same computation without using the Evaluator.  The main advantage of using the Evaluator is that you can reuse other metrics you are dependent on without having to recompute them each time, i.e it provides a local cache of metrics.

In [ ]:
mae = evaluator.metric('mae')[0]
round_trip_pnl = evaluator.metric('mae')[1] 

# Separate out positive trades from negative trades
round_trip_profit = round_trip_pnl[round_trip_pnl >= 0]
mae_profit = mae[round_trip_pnl >= 0] 

round_trip_loss = round_trip_pnl[round_trip_pnl <= 0]
mae_loss = mae[round_trip_pnl <= 0]

import plotly.graph_objects as go
fig = go.Figure()
winners = go.Scatter(name='Profitable Trade', x=mae_profit * 100, y=round_trip_profit, mode='markers', marker_color='green', marker_size=10, marker_symbol='triangle-up')
losers = go.Scatter(name='Losing Trade', x=mae_loss * 100, y=-round_trip_loss, mode='markers', marker_color='red', marker_size=10, marker_symbol='triangle-down')
fig.add_trace(winners)
fig.add_trace(losers)
fig.add_hline(y=0, opacity=0.25)
fig.add_vline(x=0, opacity=0.25)
fig.update_xaxes(title_text='Drawdown %')
fig.update_yaxes(title_text='Profit / Loss %')
if pq.has_display():
    fig.show()

It looks like a good place to put a stop loss so we keep most of the winning trades but don't take big losses might be around 0.2%

In [ ]:
strategy.df_roundtrip_trades()